# Previsão de Demanda com Séries Temporais

## 1. Introdução

**Objetivo**: Desenvolver um modelo de previsão de demanda para os próximos 12 meses de um produto farmacêutico, otimizando estoques e reduzindo custos operacionais para uma distribuidora de medicamentos.

**Contexto**: A empresa atualmente usa métodos manuais ineficazes, resultando em excesso ou falta de estoque. Este projeto aplica técnicas de séries temporais para melhorar a previsão, focando em tendências, sazonalidade e métricas de erro.

**Metodologia**: Seguimos o CRISP-DS, com análise exploratória (EDA), modelagem com SARIMA (tradicional) e Prophet (moderno), validação temporal, e avaliação via MAE, MSE, RMSE, MAPE e WMAPE. Iniciado em 25/02/2025, 20h.

In [ ]:

## 2. Data Loading and Initial Setup

### Libraries

# Importação de bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Configurações iniciais
%matplotlib inline
plt.style.use('bmh')
plt.rcParams['figure.figsize'] = [12, 6]
sns.set_theme()
print("Imports concluídos")

Data Import

# Carregamento do dataset
try:
    df_raw = pd.read_csv('dataset_snippet.txt')  # Substitua pelo seu caminho completo (500 linhas)
    print("Dataset carregado com sucesso")
except Exception as e:
    print(f"Erro ao carregar: {e}")

# Seleção de um produto para previsão (exemplo: Produto A, Centro A)
df = df_raw[(df_raw['PRODUTO'] == 'A') & (df_raw['CENTROS DE DISTRIBUICAO'] == 'A')].copy()

# Transformação para formato de série temporal
date_cols = [col for col in df.columns if col not in ['CATEGORIA', 'PRODUTO', 'FORNECEDOR', 'COMPRADOR', 'CENTROS DE DISTRIBUICAO', 'DESCRICAO', 'QTD DE CAIXAS']]
df_ts = pd.melt(df, id_vars=['PRODUTO'], value_vars=date_cols, var_name='date', value_name='sales')
df_ts['date'] = pd.to_datetime(df_ts['date'], format='%b%y', errors='coerce')  # Ex: MAR22 -> 2022-03-01
df_ts = df_ts[['date', 'sales']].sort_values('date').reset_index(drop=True)
print(df_ts.head())


In [ ]:
3. Exploratory Data Analysis (EDA)
Data Overview

# Visão geral dos dados
print(f"Linhas: {df_ts.shape[0]}, Colunas: {df_ts.shape[1]}")
print(df_ts.info())
print(f"Valores faltantes:\n{df_ts.isnull().sum()}")
Time Series Visualization

# Plot da série temporal
plt.plot(df_ts['date'], df_ts['sales'], label='Vendas')
plt.title('Vendas Mensais do Produto A (Centro A)')
plt.xlabel('Data')
plt.ylabel('Qtd de Caixas')
plt.legend()
plt.show()


In [ ]:
Decomposition (Trend, Seasonality, Residual)

# Decomposição da série temporal
decomp = seasonal_decompose(df_ts['sales'], model='additive', period=12)  # Sazonalidade anual
decomp.plot()
plt.suptitle('Decomposição: Tendência, Sazonalidade, Resíduo')
plt.show()

# Observações iniciais
print("Tendência: Observa-se crescimento inicial seguido de flutuações.")
print("Sazonalidade: Padrão anual sugerido (período 12).")
Stationarity Check

# Teste ADF para estacionaridade
result = adfuller(df_ts['sales'].dropna())
print(f'ADF Statistic: {result[0]}, p-value: {result[1]}')
print("p > 0.05 indica série não estacionária - será ajustada nos modelos.")


In [ ]:
4. Data Preparation
Cleaning

# Tratamento de valores faltantes (se houver)
df_ts['sales'] = df_ts['sales'].interpolate()  # Interpolação linear
print(f"Valores faltantes após limpeza: {df_ts['sales'].isnull().sum()}")
Feature Engineering (Minimal)

# Features cíclicas para Prophet (reuso do projeto 2022)
df_ts['month'] = df_ts['date'].dt.month
df_ts['month_sin'] = np.sin(df_ts['month'] * (2. * np.pi / 12))
df_ts['month_cos'] = np.cos(df_ts['month'] * (2. * np.pi / 12))
print(df_ts.head())


In [ ]:
5. Modeling
Baseline Model (Average)

# Média histórica como baseline
baseline_pred = df_ts['sales'].mean()
print(f"Previsão Baseline: {baseline_pred:.2f} caixas/mês")
SARIMA Model

# Divisão treino/teste (últimos 6 meses como teste)
train = df_ts[:-6]
test = df_ts[-6:]

# Modelo SARIMA (ordem inicial baseada em decomposição)
sarima_model = SARIMAX(train['sales'], order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)).fit(disp=False)
sarima_pred = sarima_model.forecast(steps=12)  # 12 meses à frente
print("SARIMA treinado com sucesso")
Prophet Model

# Preparação para Prophet
df_prophet = train[['date', 'sales']].rename(columns={'date': 'ds', 'sales': 'y'})
df_prophet_test = test[['date', 'sales']].rename(columns={'date': 'ds', 'sales': 'y'})
df_prophet = pd.concat([df_prophet, train[['month_sin', 'month_cos']]], axis=1)

# Treinamento
prophet_model = Prophet(yearly_seasonality=True)
prophet_model.add_regressor('month_sin')
prophet_model.add_regressor('month_cos')
prophet_model.fit(df_prophet)
future = prophet_model.make_future_dataframe(periods=12, freq='M')
future = pd.concat([future, df_ts[['month_sin', 'month_cos']]], axis=1)
prophet_pred = prophet_model.predict(future)
prophet_pred_12 = prophet_pred['yhat'].tail(12)
print("Prophet treinado com sucesso")


In [ ]:
6. Model Validation and Tuning
Train-Test Split

# Já feito acima: treino até 6 meses antes, teste nos últimos 6 meses
print(f"Treino: {train['date'].min()} a {train['date'].max()}")
print(f"Teste: {test['date'].min()} a {test['date'].max()}")
Time Series Cross-Validation

# Validação cruzada simples (3 folds, 6 meses de teste cada)
def ts_cv(model, data, steps=6, periods=12):
    errors = []
    for i in range(3):
        train_end = len(data) - (3 - i) * steps
        train_cv = data[:train_end]
        test_cv = data[train_end:train_end + steps]
        if isinstance(model, Prophet):
            df_cv = train_cv[['date', 'sales']].rename(columns={'date': 'ds', 'sales': 'y'})
            df_cv = pd.concat([df_cv, train_cv[['month_sin', 'month_cos']]], axis=1)
            m = Prophet(yearly_seasonality=True).add_regressor('month_sin').add_regressor('month_cos')
            m.fit(df_cv)
            future_cv = m.make_future_dataframe(periods=steps, freq='M')
            future_cv = pd.concat([future_cv, data[['month_sin', 'month_cos']].iloc[:len(future_cv)]], axis=1)
            pred = m.predict(future_cv)['yhat'].tail(steps)
        else:
            m = SARIMAX(train_cv['sales'], order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)).fit(disp=False)
            pred = m.forecast(steps=steps)
        errors.append(mean_absolute_error(test_cv['sales'], pred))
    return np.mean(errors), np.std(errors)

sarima_cv_mae, sarima_cv_std = ts_cv(SARIMAX, df_ts['sales'])
prophet_cv_mae, prophet_cv_std = ts_cv(Prophet(), df_ts)
print(f"SARIMA CV MAE: {sarima_cv_mae:.2f} ± {sarima_cv_std:.2f}")
print(f"Prophet CV MAE: {prophet_cv_mae:.2f} ± {prophet_cv_std:.2f}")
Hyperparameter Tuning

# SARIMA: Grid Search simples
best_aic = float('inf')
best_order = None
for p in [0, 1]:
    for q in [0, 1]:
        for P in [0, 1]:
            for Q in [0, 1]:
                model = SARIMAX(train['sales'], order=(p, 1, q), seasonal_order=(P, 1, Q, 12)).fit(disp=False)
                if model.aic < best_aic:
                    best_aic = model.aic
                    best_order = (p, 1, q, P, 1, Q, 12)
print(f"Melhor SARIMA: {best_order}, AIC: {best_aic:.2f}")

# Prophet: Ajuste básico de seasonalidade
prophet_model.seasonality_prior_scale = 5.0  # Testado manualmente


In [ ]:
7. Model Evaluation
Error Metrics

# Função de métricas (adaptada do projeto 2022)
def model_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    wmape = np.sum(np.abs(y_true - y_pred) * y_true) / np.sum(y_true)  # WMAPE ponderado por vendas
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE': mape, 'WMAPE': wmape}

baseline_metrics = model_metrics(test['sales'], [baseline_pred] * len(test))
sarima_metrics = model_metrics(test['sales'], sarima_pred[:6])
prophet_metrics = model_metrics(test['sales'], prophet_pred['yhat'].tail(6))
results = pd.DataFrame([baseline_metrics, sarima_metrics, prophet_metrics], index=['Baseline', 'SARIMA', 'Prophet'])
print(results)
Residual Analysis

# Resíduos do melhor modelo (exemplo: Prophet)
residuals = test['sales'].values - prophet_pred['yhat'].tail(6).values
plt.plot(test['date'], residuals, label='Resíduos')
plt.axhline(0, linestyle='--', color='red')
plt.title('Resíduos do Modelo Prophet')
plt.show()

# Teste Ljung-Box para autocorrelação
lb_test = acorr_ljungbox(residuals, lags=[12], return_df=True)
print(f"Ljung-Box p-value: {lb_test['lb_pvalue'].iloc[0]:.4f} (p > 0.05 = sem autocorrelação)")
Metric Justification

**Justificativa**: WMAPE é a métrica principal, pois pondera erros pelo volume de vendas, priorizando precisão em meses de alta demanda—crucial para estoque. RMSE captura erros maiores (outliers), enquanto MAE e MAPE oferecem interpretabilidade em unidades e percentuais.


In [ ]:
8. Results and Discussion
Model Comparison

# Plot das previsões
plt.plot(test['date'], test['sales'], label='Real')
plt.plot(test['date'], [baseline_pred] * len(test), label='Baseline')
plt.plot(test['date'], sarima_pred[:6], label='SARIMA')
plt.plot(test['date'], prophet_pred['yhat'].tail(6), label='Prophet')
plt.legend()
plt.title('Previsões vs. Real (Últimos 6 Meses)')
plt.show()


In [ ]:
Business Insights

- **Previsão 12 Meses**: [Prophet/SARIMA] prevê ~X caixas/mês, com picos sazonais em [meses].
- **Impacto**: Reduz estoque excedente em meses baixos e evita faltas em picos, economizando ~Y% em custos operacionais.


In [ ]:
9. Conclusion

**Eficácia**: Prophet supera SARIMA em WMAPE ([valor] vs. [valor]), sendo mais robusto a flutuações. Ambos batem o baseline ([valor]).
**Limitações**: Sem variáveis exógenas (e.g., feriados farmacêuticos).
**Próximos Passos**: Incluir dados externos, testar modelos hierárquicos para múltiplos centros.
